# Tutorial: Triagem com Classificação de Risco

**Público:** alunos que já viram o básico de `Environment` e `Process`.

**Pré-requisitos:** entender `yield env.timeout(...)` e a ideia de fila.

**Objetivos de aprendizagem:**

- usar `Resource` para triagem;
- usar `PriorityResource` para a fila médica;
- interpretar por que pacientes graves passam na frente sem interromper consulta já iniciada.


## Roteiro

1. Ler o cenário.
2. Entender quais recursos existem.
3. Modelar os dados dos pacientes.
4. Construir o processo de cada paciente.
5. Rodar a simulação.
6. Interpretar a ordem final dos atendimentos.


In [1]:
from __future__ import annotations

import simpy

print(f"Versão do SimPy: {simpy.__version__}")

Versão do SimPy: 4.1.1


## 1. Cenário

Imagine um pronto atendimento com:

- **1 posto de triagem**;
- **1 médico**;
- pacientes chegando em instantes diferentes;
- prioridade clínica segundo a cor de risco.

A regra central do exemplo é:

**na fila do médico, menor prioridade numérica = mais urgente = atende antes**.

Mas atenção: isso **não interrompe** um atendimento médico que já começou.


## 2. Conceitos fundamentais usados aqui

- `Resource`: representa a triagem, porque basta saber se o profissional está ocupado.
- `PriorityResource`: representa o médico, porque a ordem da fila depende do risco.
- `timeout`: representa duração de triagem e consulta.

Pergunta didática importante:

**Por que a triagem não usa prioridade?**

Porque neste exemplo a priorização acontece **depois** da classificação, ou seja, na fila médica.


In [2]:
PRIORIDADE = {
    "vermelho": 0,
    "laranja": 1,
    "amarelo": 2,
    "verde": 3,
}

PACIENTES = [
    ("P001", 0, "amarelo", 3, 12),
    ("P002", 1, "verde", 3, 8),
    ("P003", 2, "vermelho", 2, 20),
    ("P004", 4, "laranja", 3, 10),
    ("P005", 7, "amarelo", 3, 9),
]

PACIENTES

[('P001', 0, 'amarelo', 3, 12),
 ('P002', 1, 'verde', 3, 8),
 ('P003', 2, 'vermelho', 2, 20),
 ('P004', 4, 'laranja', 3, 10),
 ('P005', 7, 'amarelo', 3, 9)]

### Leitura da estrutura de dados

Cada tupla contém:

- identificador do paciente;
- instante de chegada;
- cor de risco;
- tempo de triagem;
- tempo de atendimento médico.

Esse formato é didático porque nos permite enxergar a lógica da fila sem adicionar aleatoriedade.


In [3]:
def paciente(env, nome, chegada, cor, t_triagem, t_medico, triagem, medico, resultados):
    yield env.timeout(chegada)
    print(f"{env.now:02.0f} min | {nome} chega ({cor})")

    with triagem.request() as req_triagem:
        yield req_triagem
        print(f"{env.now:02.0f} min | {nome} inicia triagem")
        yield env.timeout(t_triagem)
        fim_triagem = env.now
        print(f"{env.now:02.0f} min | {nome} termina triagem")

    with medico.request(priority=PRIORIDADE[cor]) as req_medico:
        yield req_medico
        inicio_medico = env.now
        espera_medica = inicio_medico - fim_triagem
        print(
            f"{env.now:02.0f} min | {nome} inicia médico "
            f"(espera médica={espera_medica:02.0f} min)"
        )
        yield env.timeout(t_medico)
        print(f"{env.now:02.0f} min | {nome} recebe alta")
        resultados.append((nome, cor, env.now - chegada, espera_medica))

## 3. O que esse processo está modelando?

O processo do paciente acontece em duas etapas:

1. ele disputa a triagem em fila simples;
2. depois disputa o médico em fila com prioridade.

A linha mais importante é:

`medico.request(priority=PRIORIDADE[cor])`

É aqui que a regra clínica entra no modelo.


In [4]:
def executar_simulacao():
    env = simpy.Environment()
    triagem = simpy.Resource(env, capacity=1)
    medico = simpy.PriorityResource(env, capacity=1)
    resultados = []

    for dados in PACIENTES:
        env.process(paciente(env, *dados, triagem, medico, resultados))

    env.run()
    return resultados


resultados = executar_simulacao()

00 min | P001 chega (amarelo)
00 min | P001 inicia triagem
01 min | P002 chega (verde)
02 min | P003 chega (vermelho)
03 min | P001 termina triagem
03 min | P001 inicia médico (espera médica=00 min)
03 min | P002 inicia triagem
04 min | P004 chega (laranja)
06 min | P002 termina triagem
06 min | P003 inicia triagem
07 min | P005 chega (amarelo)
08 min | P003 termina triagem
08 min | P004 inicia triagem
11 min | P004 termina triagem
11 min | P005 inicia triagem
14 min | P005 termina triagem
15 min | P001 recebe alta
15 min | P003 inicia médico (espera médica=07 min)
35 min | P003 recebe alta
35 min | P004 inicia médico (espera médica=24 min)
45 min | P004 recebe alta
45 min | P005 inicia médico (espera médica=31 min)
54 min | P005 recebe alta
54 min | P002 inicia médico (espera médica=48 min)
62 min | P002 recebe alta


In [5]:
print("Resumo final")
for nome, cor, tempo_total, espera_medica in resultados:
    print(
        f"{nome} | cor={cor:8s} | tempo total={tempo_total:02.0f} min "
        f"| espera por médico={espera_medica:02.0f} min"
    )

Resumo final
P001 | cor=amarelo  | tempo total=15 min | espera por médico=00 min
P003 | cor=vermelho | tempo total=33 min | espera por médico=07 min
P004 | cor=laranja  | tempo total=41 min | espera por médico=24 min
P005 | cor=amarelo  | tempo total=47 min | espera por médico=31 min
P002 | cor=verde    | tempo total=61 min | espera por médico=48 min


## 4. O que observar na saída

Pontos principais:

- `P002` termina a triagem cedo, mas espera muito para entrar no médico.
- `P003` chega depois, porém entra antes por ser vermelho.
- `P004` também ultrapassa `P002` por ter prioridade maior.

Moral do exemplo:

**prioridade muda a ordem da fila, não o passado da simulação**.

Ou seja: o modelo não volta no tempo e não interrompe consulta já iniciada.


## 5. Erro comum

Um erro comum é achar que `PriorityResource` funciona como interrupção automática.

Não funciona.

Se você quisesse permitir interrupção, teria de estudar `PreemptiveResource`, que é uma semântica mais forte e bem mais delicada em contexto clínico.


## 6. Exercícios

1. Troque a cor de `P005` para `vermelho` e preveja o que muda.
2. Aumente a capacidade do médico para `2` e veja como a espera muda.
3. Explique com suas palavras por que `P002` foi atendido por último.


In [6]:
# Espaço para resposta / experimento:
# 1. Altere os dados acima.
# 2. Rode novamente a célula que executa a simulação.
# 3. Compare a nova ordem de atendimento com a ordem original.

## 7. Extensão sugerida

Uma boa extensão para aula é adicionar:

- abandono de fila;
- retorno do paciente para observação;
- dois médicos com capacidades diferentes.

Isso transforma um exemplo simples de prioridade em um modelo mais próximo da operação real.
